# Ejercicio 3: Modelo Vectorial y TF-IDF

**Realizado por:** Correa Adrian  
**Fecha:** 03/05/2026

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz **TF-IDF** para un corpus pequeño (1 libro de ejemplo).
- Calcular la matriz **TF-IDF** para el corpus Gutenberg 1000 (1000 libros).
- Implementar una función de búsqueda basada en **similitud coseno**.

---

## Teoría - Fundamentos del Modelo Vectorial

### 1. Representación Vectorial
Cada documento y consulta se representa como un vector: 
$$\vec{d}, \vec{q} \in \mathbb{R}^{n}$$
donde $n$ es el número de términos distintos en el vocabulario.

### 2. Similitud del Coseno
Medida de similitud entre consulta y documento:
$$\text{sim}(\vec{q}, \vec{d}) = \frac{\vec{q} \cdot \vec{d}}{||\vec{q}|| \cdot ||\vec{d}||}$$

### 3. Componentes TF-IDF

**Frecuencia de Términos (TF):**
$$\text{tf}_{t,d} = 1 + \log(f_{t,d})$$
donde $f_{t,d}$ es la frecuencia cruda del término $t$ en el documento $d$.

**Frecuencia Inversa de Documentos (IDF):**
$$\text{idf}_{t} = \log\left(\frac{N}{df_{t}}\right)$$
donde $N$ es el número total de documentos y $df_t$ es el número de documentos que contienen el término $t$.

**Ponderación TF-IDF:**
$$w_{t,d} = \text{tf}_{t,d} \cdot \text{idf}_{t}$$

## FASE 1: PREPARACIÓN Y CARGA DE DATOS

### Paso 1: Calcular la matriz TF-IDF para un corpus pequeño

En esta práctica usaremos el archivo `01_corpus_turismo_500.txt` (ubicado en la raíz o en la carpeta `Data`) como corpus de ejemplo para la Fase 1. La celda de código siguiente lee ese archivo línea por línea (cada línea se considera un documento), calcula la matriz TF‑IDF y guarda el vectorizador y la matriz en `outputs/` para las fases posteriores.

In [42]:
import os
import re
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================================================
# PASO 1: TF-IDF PARA EL CORPUS DE TURISMO (01_corpus_turismo_500.txt)
# ============================================================================

candidates = [
    r"01_corpus_turismo_500.txt",
    r"data/01_corpus_turismo_500.txt",
    r"Data/01_corpus_turismo_500.txt",
    r"d:\\7mo semestre\\RI\\books2\\01_corpus_turismo_500.txt",
]

ruta_corpus_paso1 = next((p for p in candidates if os.path.exists(p)), None)
if ruta_corpus_paso1 is None:
    raise FileNotFoundError(f"No se encontró el archivo 01_corpus_turismo_500.txt en las rutas probadas: {candidates}")

with open(ruta_corpus_paso1, "r", encoding="utf-8", errors="ignore") as f:
    lineas = [l.strip() for l in f.readlines()]

# Filtrar líneas vacías
documentos_paso1 = [l for l in lineas if l]
if len(documentos_paso1) == 0:
    raise ValueError("El archivo está vacío o no contiene líneas útiles.")

# Lista de stopwords en español básica (puede ajustarse)
stop_words_es = [
    'el','la','los','las','de','del','que','y','a','en','un','una','por','con','para','se','su','al','lo',
    'como','más','o','pero','sus','le','ya','este','esta','son','entre','cuando','muy','sin','sobre',
    'también','me','hasta','hay','donde','todo','uno','ni','otros','ese','eso','antes','durante'
]

vectorizador_paso1 = TfidfVectorizer(
    stop_words=stop_words_es,
    token_pattern=r"(?u)\b[a-záéíóúñ]{2,}\b",
    min_df=1,
    max_df=0.95,
    norm='l2',
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=True
)

# Entrenar vectorizador y calcular TF-IDF
vectorizador_paso1.fit(documentos_paso1)

# Calcular df y forzar idf = log(N/df)
tfidf_counts = (vectorizador_paso1.transform(documentos_paso1) > 0).astype(int)
df_counts = np.asarray(tfidf_counts.sum(axis=0)).ravel()
N_docs = len(documentos_paso1)
# Evitar división por cero (si df_counts contiene ceros, setear a 1)
df_counts_safe = np.where(df_counts == 0, 1, df_counts)
idf_target = np.log(N_docs / df_counts_safe)
vectorizador_paso1.idf_ = idf_target
if hasattr(vectorizador_paso1, '_tfidf'):
    vectorizador_paso1._tfidf.idf_ = idf_target

matriz_tfidf_paso1 = vectorizador_paso1.transform(documentos_paso1)
vocabulario_paso1 = vectorizador_paso1.get_feature_names_out()

print("\n" + "=" * 80)
print("PASO 1: MATRIZ TF-IDF - CORPUS TURISMO (01_corpus_turismo_500.txt)")
print("=" * 80)
print(f"Archivo utilizado: {ruta_corpus_paso1}")
print(f"Documentos (líneas no vacías): {N_docs}")
print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf_paso1.shape[0]} documentos × {matriz_tfidf_paso1.shape[1]} términos")
print(f"Elementos no-cero: {matriz_tfidf_paso1.nnz}")

print("\nPrimeros 25 términos del vocabulario:")
print(", ".join(vocabulario_paso1[:25]))

# Mostrar una vista pequeña de la matriz como DataFrame
vista = pd.DataFrame(
    matriz_tfidf_paso1[:5].toarray(),
    columns=vocabulario_paso1
)
print("\nVista parcial de la matriz TF-IDF (primeros 5 documentos):")
print(vista.iloc[:, :15].head())

# Guardar para fases posteriores
from joblib import dump
import json
os.makedirs('outputs', exist_ok=True)
with open(os.path.join('outputs', 'tfidf_vocab_paso1.json'), 'w', encoding='utf-8') as f:
    json.dump(vocabulario_paso1.tolist(), f, ensure_ascii=False)
dump(vectorizador_paso1, os.path.join('outputs', 'tfidf_vectorizer_paso1.joblib'))
dump(matriz_tfidf_paso1, os.path.join('outputs', 'tfidf_matrix_paso1.joblib'))
print('\nVectorizador y matriz guardados en carpeta outputs/')


PASO 1: MATRIZ TF-IDF - CORPUS TURISMO (01_corpus_turismo_500.txt)
Archivo utilizado: 01_corpus_turismo_500.txt
Documentos (líneas no vacías): 500
Dimensiones de la matriz TF-IDF: 500 documentos × 98 términos
Elementos no-cero: 3887

Primeros 25 términos del vocabulario:
agua, amazonía, arquitectura, artesanía, atrae, atraen, auténtico, aventura, aves, avistamiento, baños, biodiversidad, cajas, caminatas, canopy, centro, colonial, color, conecta, conocido, cotopaxi, cuenca, deportes, deslumbra, destaca

Vista parcial de la matriz TF-IDF (primeros 5 documentos):
   agua  amazonía  arquitectura  artesanía     atrae  atraen  auténtico  \
0   0.0       0.0           0.0    0.39841  0.000000     0.0        0.0   
1   0.0       0.0           0.0    0.00000  0.000000     0.0        0.0   
2   0.0       0.0           0.0    0.00000  0.402682     0.0        0.0   
3   0.0       0.0           0.0    0.00000  0.000000     0.0        0.0   
4   0.0       0.0           0.0    0.00000  0.000000    

### Paso 2: Construir el corpus Gutenberg 1000

**INFORMACIÓN DE ARCHIVOS:**
- **Ruta:** `D:\7mo semestre\RI\books2\Data\`
- **Cantidad de archivos:** 1000 archivos `.txt` individuales (tomamos los primeros 1000 en orden alfabético)
- **Tipo:** Libros de Project Gutenberg (dominio público)

In [43]:
import glob
import time

print("\n" + "=" * 80)
print("PASO 2: CONSTRUIR EL CORPUS GUTENBERG 1000")
print("=" * 80)

print("\nINFORMACIÓN DE ARCHIVOS:")
print("-" * 80)
print("Ruta:                              D:\\7mo semestre\\RI\\books2\\Data\\")
print("Tipo de archivos:                  .txt individuales (1 libro por archivo)")
print("Cantidad total de archivos:        1000 (primeros 1000 en orden alfabético)")
print("-" * 80 + "\n")

ruta_corpus_1000 = r"d:\7mo semestre\RI\books2\Data"
archivos_todos = sorted(glob.glob(os.path.join(ruta_corpus_1000, "*.txt")))
archivos_1000 = archivos_todos[:1000]

print(f"Total de archivos encontrados en Data: {len(archivos_todos)}")
print(f"Archivos que se usarán en el corpus Gutenberg 1000: {len(archivos_1000)}")
print("\nPrimeros 10 archivos:")
for indice, archivo in enumerate(archivos_1000[:10], 1):
    print(f"  {indice:2d}. {os.path.basename(archivo)}")
print("  ...\n")


def limpiar_texto(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans("", "", string.punctuation))
    texto = re.sub(r"\d+", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


stop_words_es = [
    "el", "la", "de", "que", "y", "a", "en", "un", "ser", "se", "no", "haber",
    "por", "con", "su", "para", "es", "al", "lo", "como", "más", "o", "pero",
    "sus", "le", "ya", "fue", "este", "ha", "porque", "esta", "son", "entre",
    "está", "cuando", "muy", "sin", "sobre", "tiene", "también", "me", "hasta",
    "hay", "donde", "han", "quien", "están", "estado", "desde", "todo", "nos",
    "durante", "todos", "uno", "les", "ni", "contra", "otros", "fueron", "ese",
    "eso", "había", "ante", "ellos", "esto", "antes"
]

print("Cargando documentos...")
tiempo_inicio = time.time()
documentos_1000 = []
nombres_docs_1000 = []

for indice, archivo in enumerate(archivos_1000, 1):
    try:
        with open(archivo, "r", encoding="utf-8", errors="ignore") as f:
            contenido = f.read()
        texto_limpio = limpiar_texto(contenido[:25000])
        if len(texto_limpio) > 100:
            documentos_1000.append(texto_limpio)
            nombres_docs_1000.append(os.path.basename(archivo))
        if len(documentos_1000) % 250 == 0:
            print(f"  ✓ {len(documentos_1000)} documentos procesados")
    except Exception:
        pass

tiempo_carga_1000 = time.time() - tiempo_inicio

print(f"\n{'=' * 80}")
print("ESTADÍSTICAS DEL CORPUS GUTENBERG 1000")
print(f"{'=' * 80}")
print(f"  Total de documentos cargados: {len(documentos_1000)}")
print(f"  Tiempo de carga: {tiempo_carga_1000:.2f}s")
palabras_por_doc = [len(doc.split()) for doc in documentos_1000]
print(f"  Palabras promedio/documento: {int(np.mean(palabras_por_doc))}")
print(f"  Palabras min/max: {int(np.min(palabras_por_doc))} / {int(np.max(palabras_por_doc))}")


PASO 2: CONSTRUIR EL CORPUS GUTENBERG 1000

INFORMACIÓN DE ARCHIVOS:
--------------------------------------------------------------------------------
Ruta:                              D:\7mo semestre\RI\books2\Data\
Tipo de archivos:                  .txt individuales (1 libro por archivo)
Cantidad total de archivos:        1000 (primeros 1000 en orden alfabético)
--------------------------------------------------------------------------------

Total de archivos encontrados en Data: 1000
Archivos que se usarán en el corpus Gutenberg 1000: 1000

Primeros 10 archivos:
   1. 20 poemas para ser leídos en el tranvía.txt
   2. 40 years  40 años  40 ans.txt
   3. 7 de julio.txt
   4. A Doll's House  a play.txt
   5. A First Spanish Reader.txt
   6. A Flor De Piel Frases.txt
   7. A History of Magic and Experimental Science, Volume 1 (of 2) During the First Thirteen Centuries of .txt
   8. A Modest Proposal For preventing the children of poor people in Ireland, from being a burden on thei.tx

### Paso 3: Calcular la matriz TF-IDF para el corpus Gutenberg 1000

In [32]:
print("\n" + "=" * 80)
print("PASO 3: CALCULAR LA MATRIZ TF-IDF PARA EL CORPUS GUTENBERG 1000")
print("=" * 80)

vectorizador = TfidfVectorizer(
    max_features=5000,
    stop_words=stop_words_es,
    token_pattern=r"(?u)\b[a-záéíóúñ]{3,}\b",
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    norm="l2",
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=True,
)

tiempo_inicio = time.time()
# Ajuste para que IDF coincida con la diapositiva: idf = log(N/df)
vectorizador.fit(documentos_1000)

# Calcular df (número de documentos que contienen cada término)
tfidf_counts_1000 = (vectorizador.transform(documentos_1000) > 0).astype(int)
df_counts_1000 = np.asarray(tfidf_counts_1000.sum(axis=0)).ravel()
N_docs_1000 = len(documentos_1000)
idf_target_1000 = np.log(N_docs_1000 / df_counts_1000)

# Sobrescribir idf en el vectorizador y su transformador interno
vectorizador.idf_ = idf_target_1000
if hasattr(vectorizador, '_tfidf'):
    vectorizador._tfidf.idf_ = idf_target_1000

# Transformar con la idf personalizada
matriz_tfidf = vectorizador.transform(documentos_1000)
tiempo_vec = time.time() - tiempo_inicio

print(f"\n{'=' * 80}")
print("MATRIZ TF-IDF GENERADA")
print(f"{'=' * 80}")
print(f"  Dimensiones: {matriz_tfidf.shape[0]} documentos × {matriz_tfidf.shape[1]} términos")
print(f"  Elementos no-cero: {matriz_tfidf.nnz:,}")
densidad = (matriz_tfidf.nnz / (matriz_tfidf.shape[0] * matriz_tfidf.shape[1])) * 100
print(f"  Densidad: {densidad:.4f}%")
print(f"  Tiempo de vectorización: {tiempo_vec:.2f}s")

nombres_features = vectorizador.get_feature_names_out()
print(f"\n  Vocabulario total: {len(nombres_features)} términos únicos")
print(f"  Primeros 30 términos: {', '.join(nombres_features[:30])}")

# Vista explícita de la submatriz (primeros 5 docs x primeros 15 términos)
import pandas as pd
vista_tfidf_1000 = pd.DataFrame(
    matriz_tfidf[:5, :15].toarray(),
    columns=nombres_features[:15],
    index=[f"doc_{i+1}: {nombres_docs_1000[i]}" for i in range(5)]
)
print("\nSubmatriz TF-IDF (primeros 5 documentos x primeros 15 términos):")
print(vista_tfidf_1000.round(4))

# Tabla explícita: términos más importantes por documento (primeros 3 docs)
print("\nTop 10 términos por documento (docs 1 a 3):")
for doc_i in range(3):
    fila = matriz_tfidf[doc_i].toarray().ravel()
    idx_top = fila.argsort()[::-1]
    idx_top = [j for j in idx_top if fila[j] > 0][:10]
    tabla_doc = pd.DataFrame({
        'termino': [nombres_features[j] for j in idx_top],
        'tfidf': [fila[j] for j in idx_top]
    })
    print(f"\nDocumento {doc_i + 1}: {nombres_docs_1000[doc_i]}")
    print(tabla_doc.to_string(index=False, formatters={'tfidf': '{:.4f}'.format}))

# Resumen de no-cero por documento (primeros 5)
nnz_docs = np.asarray((matriz_tfidf[:5] > 0).sum(axis=1)).ravel()
print("\nNúmero de términos no-cero por documento (primeros 5 docs):")
for i, c in enumerate(nnz_docs, 1):
    print(f"  doc_{i} ({nombres_docs_1000[i-1]}): {int(c)}")


PASO 3: CALCULAR LA MATRIZ TF-IDF PARA EL CORPUS GUTENBERG 1000

MATRIZ TF-IDF GENERADA
  Dimensiones: 1000 documentos × 5000 términos
  Elementos no-cero: 689,774
  Densidad: 13.7955%
  Tiempo de vectorización: 23.55s

  Vocabulario total: 5000 términos únicos
  Primeros 30 términos: abad, abajo, abandonado, abandonar, abandono, abierta, abierto, abismo, able, abogado, about, about the, above, abre, abriendo, abrigo, abril, abrir, abrió, absoluta, absolutamente, absoluto, abuela, abuelo, abundancia, abundante, acaba, acababa, acabado, acabar

Submatriz TF-IDF (primeros 5 documentos x primeros 15 términos):
                                                    abad   abajo  abandonado  \
doc_1: 20 poemas para ser leídos en el tranvía.txt   0.0  0.0000         0.0   
doc_2: 40 years  40 años  40 ans.txt                 0.0  0.0000         0.0   
doc_3: 7 de julio.txt                                0.0  0.0000         0.0   
doc_4: A Doll's House  a play.txt                    0.0  0.0000

### Análisis: Términos más relevantes en el corpus

In [33]:
print("\n" + "="*80)
print("ANÁLISIS DE VOCABULARIO Y TÉRMINOS RELEVANTES")
print("="*80)

# Calcular TF-IDF promedio para cada término
tfidf_promedio = np.asarray(matriz_tfidf.mean(axis=0)).ravel()
indices_top = np.argsort(tfidf_promedio)[::-1][:50]

print(f"\nEstadísticas TF-IDF:")
print(f"  Mínimo: {matriz_tfidf.min():.6f}")
print(f"  Máximo: {matriz_tfidf.max():.6f}")
print(f"  Promedio: {tfidf_promedio.mean():.6f}")
print(f"  Desviación estándar: {tfidf_promedio.std():.6f}")

print(f"\nTop 30 términos por relevancia TF-IDF promedio:")
print(f"{'Rank':<5} {'Término':<25} {'TF-IDF Promedio':<20}")
print("-" * 50)

for i, idx in enumerate(indices_top[:30], 1):
    termino = nombres_features[idx]
    score = tfidf_promedio[idx]
    print(f"{i:<5} {termino:<25} {score:>18.6f}")


ANÁLISIS DE VOCABULARIO Y TÉRMINOS RELEVANTES

Estadísticas TF-IDF:
  Mínimo: 0.000000
  Máximo: 0.661662
  Promedio: 0.004496
  Desviación estándar: 0.001765

Top 30 términos por relevancia TF-IDF promedio:
Rank  Término                   TF-IDF Promedio     
--------------------------------------------------
1     the                                 0.016036
2     and                                 0.015979
3     usted                               0.014305
4     that                                0.012794
5     for                                 0.012363
6     mas                                 0.011766
7     with                                0.011757
8     fué                                 0.011489
9     his                                 0.011186
10    not                                 0.010870
11    don                                 0.010789
12    rey                                 0.010676
13    have                                0.010621
14    doña              

### Paso 4: Programar una función `buscar()` para el corpus Gutenberg 1000

In [34]:
print("\n" + "="*80)
print("PASO 4: FUNCIÓN DE BÚSQUEDA - SIMILITUD COSENO (VERSIÓN ROBUSTA)")
print("="*80)

from sklearn.metrics.pairwise import cosine_similarity

def buscar(consulta, corpus_tfidf, vectorizer, nombres_documentos, documentos_texto=None, top_k=10):
    """
    Busca documentos relevantes usando similitud coseno.

    Mejoras de robustez:
    - Valida consultas vacías.
    - Detecta consultas sin términos presentes en el vocabulario.
    - Devuelve métricas interpretables (similitud, términos coincidentes y fragmento).

    Retrocompatibilidad:
    - Si `documentos_texto` no se pasa, intenta usar `documentos_1000` del entorno.
    """

    columnas = [
        'Rank',
        'Documento',
        'Similitud',
        'Similitud (%)',
        'Terminos coincidentes',
        'Fragmento'
    ]

    if documentos_texto is None:
        documentos_texto = globals().get('documentos_1000', [])

    # Paso 1: validar y limpiar consulta
    if consulta is None or not str(consulta).strip():
        return pd.DataFrame(columns=columnas)

    consulta_limpia = limpiar_texto(consulta)
    if not consulta_limpia:
        return pd.DataFrame(columns=columnas)

    # Paso 2: vectorizar consulta
    vector_consulta = vectorizer.transform([consulta_limpia])
    query_idx = vector_consulta.indices

    # Si no hay términos de consulta en el vocabulario, no hay resultados útiles
    if len(query_idx) == 0:
        return pd.DataFrame(columns=columnas)

    # Paso 3: similitud coseno
    similitudes = cosine_similarity(vector_consulta, corpus_tfidf)[0]

    # Paso 4: ordenar por relevancia
    indices_ordenados = np.argsort(similitudes)[::-1]

    # Paso 5: construir resultados
    resultados = []
    rank = 1
    for idx in indices_ordenados:
        if len(resultados) >= top_k:
            break
        if similitudes[idx] <= 0:
            continue

        doc_indices = corpus_tfidf[idx].indices
        n_match = int(np.intersect1d(query_idx, doc_indices).size)

        texto_doc = ''
        if idx < len(documentos_texto):
            texto_doc = str(documentos_texto[idx]).replace('\n', ' ')
        fragmento = texto_doc[:180] + ('...' if len(texto_doc) > 180 else '')

        resultados.append({
            'Rank': rank,
            'Documento': nombres_documentos[idx],
            'Similitud': float(similitudes[idx]),
            'Similitud (%)': f"{similitudes[idx]*100:.2f}%",
            'Terminos coincidentes': n_match,
            'Fragmento': fragmento
        })
        rank += 1

    return pd.DataFrame(resultados, columns=columnas)


# Demostración: ejecutar búsquedas de prueba
print("\nEjecutando búsquedas en el corpus Gutenberg 1000:\n")

consultas_demo = [
    'amor pasión romance',
    'misterio intriga secreto',
    'aventura viaje exploración',
    'muerte tragedia dolor',
    'esperanza futuro vida'
]

for i, consulta in enumerate(consultas_demo, 1):
    print(f"\n{'='*80}")
    print(f"CONSULTA {i}: \"{consulta}\"")
    print(f"{'='*80}")

    resultados = buscar(
        consulta,
        matriz_tfidf,
        vectorizador,
        nombres_docs_1000,
        documentos_1000,
        top_k=8
    )

    if len(resultados) > 0:
        print(resultados[['Rank', 'Documento', 'Similitud', 'Similitud (%)', 'Terminos coincidentes']].to_string(index=False))
        print("\nPrimer resultado - fragmento:")
        print(resultados.iloc[0]['Fragmento'])
    else:
        print("  [Sin resultados relevantes o consulta fuera de vocabulario]")

    print()


PASO 4: FUNCIÓN DE BÚSQUEDA - SIMILITUD COSENO (VERSIÓN ROBUSTA)

Ejecutando búsquedas en el corpus Gutenberg 1000:


CONSULTA 1: "amor pasión romance"
 Rank                                                                                       Documento  Similitud Similitud (%)  Terminos coincidentes
    1                                 Las cien mejores poesías (lí­ricas) de la lengua castellana.txt   0.108368        10.84%                      2
    2                                                                            Romancero gitano.txt   0.098185         9.82%                      2
    3                                                                          Novelas ejemplares.txt   0.097383         9.74%                      1
    4                                                                             Libro de poemas.txt   0.092912         9.29%                      3
    5                                                                            Novelas y teatro

In [36]:
print("\n" + "="*80)
print("BÚSQUEDAS AVANZADAS Y VALIDACIÓN")
print("="*80)

# Comparación: Autoconsulta (documento como consulta)
print("\nValidación 1: Autoconsulta (documento comparado consigo mismo)")
print("-" * 80)

idx_prueba = 100
doc_prueba = documentos_1000[idx_prueba]
nombre_prueba = nombres_docs_1000[idx_prueba]

vector_doc = vectorizador.transform([doc_prueba])
similitudes = cosine_similarity(vector_doc, matriz_tfidf)[0]

indices_similares = np.argsort(similitudes)[::-1][1:6]

print(f"\nDocumento de referencia: {nombre_prueba}")
print(f"Primeras palabras: {' '.join(doc_prueba.split()[:15])}...\n")
print("Documentos más similares:")

for rank, idx in enumerate(indices_similares, 1):
    print(f"  {rank}. {nombres_docs_1000[idx]:<60} -> {similitudes[idx]*100:6.2f}%")

# Comparación: Búsquedas complejas
print(f"\n{'='*80}")
print("Validación 2: Búsquedas con diferentes niveles de especificidad")
print("="*80)

consultas_variadas = [
    'libro',  # Muy general
    'príncipe reino',  # Moderada
    'príncipe dinamarca tragedia shakespeare',  # Específica
]

for consulta in consultas_variadas:
    resultados = buscar(consulta, matriz_tfidf, vectorizador, nombres_docs_1000, top_k=3)
    print(f"\nConsulta: \"{consulta}\"")
    if len(resultados) > 0:
        similitud_max = resultados['Similitud'].max()
        print(f"  Relevancia máxima: {similitud_max*100:.2f}%")
        print(f"  Resultado top: {resultados.iloc[0]['Documento']}")
    else:
        print("  [Sin resultados]")


BÚSQUEDAS AVANZADAS Y VALIDACIÓN

Validación 1: Autoconsulta (documento comparado consigo mismo)
--------------------------------------------------------------------------------

Documento de referencia: Comentario del coronel Francisco Verdugo, de la guerra de Frisia, en xiv años que fue gobernador y c.txt
Primeras palabras: coleccion de libros españoles raros ó curiosos tomo segundo comentario del coronel francisco verdugo de...

Documentos más similares:
  1. Expedición de Catalanes y Aragoneses al Oriente.txt          ->  31.24%
  2. Historia de la Conquista de Mexico, Volume 2 (of 3) Poblacion y Progresos de la America Septentriona.txt ->  30.25%
  3. Historia de las Indias (vol. 5 de 5).txt                     ->  28.59%
  4. Verdadera historia de los sucesos de la conquista de la Nueva-España (3 de 3).txt ->  28.30%
  5. Historia de las Indias (vol. 3 de 5).txt                     ->  28.13%

Validación 2: Búsquedas con diferentes niveles de especificidad

Consulta: "libro"
  R

In [37]:
print("\n" + "="*80)
print("COMPARACIÓN: CORPUS DE 500 vs 1000 DOCUMENTOS")
print("="*80)

# Esta celda es autocontenida: crea un corpus de 500 documentos a partir
# del mismo corpus Gutenberg 1000 para evitar depender de variables antiguas.
documentos_500 = documentos_1000[:500]
nombres_docs_500 = nombres_docs_1000[:500]
vectorizador_500 = TfidfVectorizer(
    max_features=5000,
    stop_words=stop_words_es,
    token_pattern=r"(?u)\b[a-záéíóúñ]{3,}\b",
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    norm="l2",
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=True,
)

vectorizador_500.fit(documentos_500)
tfidf_counts_500 = (vectorizador_500.transform(documentos_500) > 0).astype(int)
df_counts_500 = np.asarray(tfidf_counts_500.sum(axis=0)).ravel()
N_docs_500 = len(documentos_500)
idf_target_500 = np.log(N_docs_500 / df_counts_500)
vectorizador_500.idf_ = idf_target_500
if hasattr(vectorizador_500, '_tfidf'):
    vectorizador_500._tfidf.idf_ = idf_target_500
matriz_tfidf_500 = vectorizador_500.transform(documentos_500)

# Analizar diferencias en matrices
print(f"\nCaracterísticas de los dos corpus:")
print(f"{'Métrica':<30} {'500 docs':<20} {'1000 docs':<20}")
print("-" * 70)
print(f"{'Dimensión matriz':<30} {matriz_tfidf_500.shape[1]:<20} {matriz_tfidf.shape[1]:<20}")
nnz_500 = f"{matriz_tfidf_500.nnz:,}"
nnz_1000 = f"{matriz_tfidf.nnz:,}"
print(f"{'Elementos no-cero':<30} {nnz_500:<20} {nnz_1000:<20}")
densidad_500_calc = (matriz_tfidf_500.nnz / (matriz_tfidf_500.shape[0] * matriz_tfidf_500.shape[1])) * 100
densidad_1000_calc = (matriz_tfidf.nnz / (matriz_tfidf.shape[0] * matriz_tfidf.shape[1])) * 100
print(f"{'Densidad (%)':<30} {densidad_500_calc:.4f}%{'':<13} {densidad_1000_calc:.4f}%")

# Buscar la misma consulta en ambos corpus
consulta_test = 'aventura historia'

print(f"\n\nBúsqueda de \"{consulta_test}\" en ambos corpus:\n")

resultados_500 = buscar(consulta_test, matriz_tfidf_500, vectorizador_500, nombres_docs_500, top_k=5)
resultados_1000 = buscar(consulta_test, matriz_tfidf, vectorizador, nombres_docs_1000, top_k=5)

print("En corpus de 500 documentos:")
if len(resultados_500) > 0:
    print(resultados_500[['Rank', 'Similitud (%)']].to_string(index=False))
    print(f"Relevancia promedio: {resultados_500['Similitud'].mean()*100:.2f}%")
else:
    print("  [Sin resultados]")

print(f"\nEn corpus de 1000 documentos:")
if len(resultados_1000) > 0:
    print(resultados_1000[['Rank', 'Similitud (%)']].to_string(index=False))
    print(f"Relevancia promedio: {resultados_1000['Similitud'].mean()*100:.2f}%")
else:
    print("  [Sin resultados]")


COMPARACIÓN: CORPUS DE 500 vs 1000 DOCUMENTOS

Características de los dos corpus:
Métrica                        500 docs             1000 docs           
----------------------------------------------------------------------
Dimensión matriz               5000                 5000                
Elementos no-cero              353,553              689,774             
Densidad (%)                   14.1421%              13.7955%


Búsqueda de "aventura historia" en ambos corpus:

En corpus de 500 documentos:
 Rank Similitud (%)
    1        13.98%
    2         9.90%
    3         8.03%
    4         6.49%
    5         6.32%
Relevancia promedio: 8.94%

En corpus de 1000 documentos:
 Rank Similitud (%)
    1        14.97%
    2        10.89%
    3         8.42%
    4         8.18%
    5         7.75%
Relevancia promedio: 10.04%


In [38]:
# Compact verification of Paso 4: show top-8 results for each demo query
consultas_compactas = [
    'amor pasión romance',
    'misterio intriga secreto',
    'aventura viaje exploración',
    'muerte tragedia dolor',
    'esperanza futuro vida'
]

for consulta in consultas_compactas:
    print('\n' + '='*80)
    print(f'CONSULTA: "{consulta}"')
    print('='*80)
    resultados = buscar(consulta, matriz_tfidf, vectorizador, nombres_docs_1000, top_k=8)
    if resultados.empty:
        print('  [Sin resultados]')
        continue
    for _, row in resultados.iterrows():
        print(f"{int(row['Rank']):>3}  {row['Documento']:<80} {row['Similitud']:.6f}  {row['Similitud (%)']}")


CONSULTA: "amor pasión romance"
  1  Las cien mejores poesías (lí­ricas) de la lengua castellana.txt                  0.108368  10.84%
  2  Romancero gitano.txt                                                             0.098185  9.82%
  3  Novelas ejemplares.txt                                                           0.097383  9.74%
  4  Libro de poemas.txt                                                              0.092912  9.29%
  5  Novelas y teatro.txt                                                             0.087268  8.73%
  6  Historia de la lengua y literatura castellana, Tomo 1  $b Desde los orígenes hasta Carlos V.txt 0.082107  8.21%
  7  Comedias El remedio en la desdicha; El mejor alcalde, el rey.txt                 0.078996  7.90%
  8  Novelas ejemplares y amorosas.txt                                                0.071981  7.20%

CONSULTA: "misterio intriga secreto"
  1  El tesoro misterioso.txt                                                         0.133804  1

In [44]:
# Verificación compacta de Fase 1
import numpy as np

print('=' * 70)
print('VERIFICACION FASE 1')
print('=' * 70)
print(f'ruta_corpus_paso1: {ruta_corpus_paso1}')
print(f'N_docs: {N_docs}')
print(f'shape: {matriz_tfidf_paso1.shape}')
print(f'nnz: {matriz_tfidf_paso1.nnz}')
print(f'vocab_size: {len(vocabulario_paso1)}')

# Comprobaciones de consistencia
ok_docs = (N_docs == 500)
ok_shape = (matriz_tfidf_paso1.shape[0] == N_docs)
ok_vocab = (matriz_tfidf_paso1.shape[1] == len(vocabulario_paso1))
ok_nnz = (matriz_tfidf_paso1.nnz > 0)

print(f'ok_docs_500: {ok_docs}')
print(f'ok_shape_consistente: {ok_shape}')
print(f'ok_vocab_consistente: {ok_vocab}')
print(f'ok_nnz_mayor_que_0: {ok_nnz}')

# Densidad de la matriz (debe ser baja en TF-IDF)
densidad = matriz_tfidf_paso1.nnz / (matriz_tfidf_paso1.shape[0] * matriz_tfidf_paso1.shape[1])
print(f'densidad: {densidad:.6f} ({densidad*100:.3f}%)')

# Top 10 términos más frecuentes por presencia documental
df_terms = np.asarray((matriz_tfidf_paso1 > 0).sum(axis=0)).ravel()
idx_top_df = np.argsort(df_terms)[::-1][:10]
print('\nTop 10 terminos por presencia en documentos:')
for i in idx_top_df:
    print(f"  {vocabulario_paso1[i]} -> df={int(df_terms[i])}")

VERIFICACION FASE 1
ruta_corpus_paso1: 01_corpus_turismo_500.txt
N_docs: 500
shape: (500, 98)
nnz: 3887
vocab_size: 98
ok_docs_500: True
ok_shape_consistente: True
ok_vocab_consistente: True
ok_nnz_mayor_que_0: True
densidad: 0.079327 (7.933%)

Top 10 terminos por presencia en documentos:
  ideal -> df=142
  feriado -> df=138
  es -> df=119
  próximo -> df=118
  perfecto -> df=100
  inolvidable -> df=96
  experiencia -> df=96
  visitar -> df=92
  lugar -> df=92
  montañita -> df=74


In [40]:
# Verificación compacta de Paso 2
import os
import numpy as np

print('=' * 70)
print('VERIFICACION PASO 2')
print('=' * 70)

ok_total_archivos = (len(archivos_todos) == 1000)
ok_archivos_usados = (len(archivos_1000) == 1000)
ok_docs_cargados = (len(documentos_1000) == 1000)
ok_nombres_docs = (len(nombres_docs_1000) == len(documentos_1000))

print(f'total_archivos_en_data: {len(archivos_todos)}')
print(f'archivos_usados_gutenberg_1000: {len(archivos_1000)}')
print(f'documentos_cargados: {len(documentos_1000)}')
print(f'tiempo_carga_s: {tiempo_carga_1000:.2f}')

prom = int(np.mean(palabras_por_doc))
minimo = int(np.min(palabras_por_doc))
maximo = int(np.max(palabras_por_doc))
print(f'palabras_promedio: {prom}')
print(f'palabras_min_max: {minimo} / {maximo}')

print(f'ok_total_archivos_1000: {ok_total_archivos}')
print(f'ok_archivos_usados_1000: {ok_archivos_usados}')
print(f'ok_documentos_cargados_1000: {ok_docs_cargados}')
print(f'ok_nombres_docs_consistente: {ok_nombres_docs}')

print('\nPrimeros 5 archivos utilizados:')
for i, nombre in enumerate(nombres_docs_1000[:5], 1):
    print(f'  {i}. {nombre}')

VERIFICACION PASO 2
total_archivos_en_data: 1000
archivos_usados_gutenberg_1000: 1000
documentos_cargados: 1000
tiempo_carga_s: 9.82
palabras_promedio: 3924
palabras_min_max: 402 / 4803
ok_total_archivos_1000: True
ok_archivos_usados_1000: True
ok_documentos_cargados_1000: True
ok_nombres_docs_consistente: True

Primeros 5 archivos utilizados:
  1. 20 poemas para ser leídos en el tranvía.txt
  2. 40 years  40 años  40 ans.txt
  3. 7 de julio.txt
  4. A Doll's House  a play.txt
  5. A First Spanish Reader.txt


In [41]:
# Verificación compacta de Paso 3
import numpy as np

print('=' * 70)
print('VERIFICACION PASO 3')
print('=' * 70)

shape_docs, shape_terms = matriz_tfidf.shape
dens_calc = (matriz_tfidf.nnz / (shape_docs * shape_terms)) * 100

print(f'dimensiones: {shape_docs} x {shape_terms}')
print(f'nnz: {matriz_tfidf.nnz:,}')
print(f'densidad_calc: {dens_calc:.4f}%')
print(f'tiempo_vectorizacion_s: {tiempo_vec:.2f}')
print(f'vocab_total: {len(nombres_features)}')
print(f'primeros_10_terminos: {", ".join(nombres_features[:10])}')

ok_docs = (shape_docs == 1000)
ok_terms = (shape_terms == 5000)
ok_vocab = (len(nombres_features) == 5000)
ok_dens = abs(dens_calc - 13.7955) < 0.01

print(f'ok_docs_1000: {ok_docs}')
print(f'ok_terms_5000: {ok_terms}')
print(f'ok_vocab_5000: {ok_vocab}')
print(f'ok_densidad_aprox_13_7955: {ok_dens}')

VERIFICACION PASO 3
dimensiones: 1000 x 5000
nnz: 689,774
densidad_calc: 13.7955%
tiempo_vectorizacion_s: 23.55
vocab_total: 5000
primeros_10_terminos: abad, abajo, abandonado, abandonar, abandono, abierta, abierto, abismo, able, abogado
ok_docs_1000: True
ok_terms_5000: True
ok_vocab_5000: True
ok_densidad_aprox_13_7955: True


### Pruebas automáticas de casos límite (`buscar`)

In [35]:
# Edge-case tests para validar robustez de buscar()

casos = [
    {
        'nombre': 'Consulta vacía',
        'consulta': '',
        'espera_vacio': True
    },
    {
        'nombre': 'Solo espacios',
        'consulta': '   ',
        'espera_vacio': True
    },
    {
        'nombre': 'Solo stopwords',
        'consulta': 'de la que y en el',
        'espera_vacio': True
    },
    {
        'nombre': 'Fuera de vocabulario',
        'consulta': 'qwertyuiop asdfghjkl zxcvbnm',
        'espera_vacio': True
    },
    {
        'nombre': 'Consulta normal',
        'consulta': 'amor tragedia vida',
        'espera_vacio': False
    }
]

print('=' * 80)
print('PRUEBAS AUTOMATICAS - EDGE CASES DE buscar()')
print('=' * 80)

resumen = []
for i, caso in enumerate(casos, 1):
    resultados = buscar(
        caso['consulta'],
        matriz_tfidf,
        vectorizador,
        nombres_docs_1000,
        documentos_1000,
        top_k=5
    )

    esta_vacio = resultados.empty
    ok = (esta_vacio == caso['espera_vacio'])

    resumen.append({
        'Caso': caso['nombre'],
        'Consulta': caso['consulta'],
        'Esperado vacio': caso['espera_vacio'],
        'Obtenido vacio': esta_vacio,
        'Num resultados': int(len(resultados)),
        'Estado': 'OK' if ok else 'FAIL'
    })

    print(f"{i}. {caso['nombre']}: {'OK' if ok else 'FAIL'}")

resumen_df = pd.DataFrame(resumen)
print('\nResumen de pruebas:')
print(resumen_df.to_string(index=False))

n_ok = int((resumen_df['Estado'] == 'OK').sum())
n_total = int(len(resumen_df))
print(f"\nResultado global: {n_ok}/{n_total} pruebas OK")

if n_ok == n_total:
    print('Todas las pruebas de robustez pasaron correctamente.')
else:
    print('Hay pruebas fallidas; revisar comportamiento de buscar().')

PRUEBAS AUTOMATICAS - EDGE CASES DE buscar()
1. Consulta vacía: OK
2. Solo espacios: OK
3. Solo stopwords: OK
4. Fuera de vocabulario: OK
5. Consulta normal: OK

Resumen de pruebas:
                Caso                     Consulta  Esperado vacio  Obtenido vacio  Num resultados Estado
      Consulta vacía                                         True            True               0     OK
       Solo espacios                                         True            True               0     OK
      Solo stopwords            de la que y en el            True            True               0     OK
Fuera de vocabulario qwertyuiop asdfghjkl zxcvbnm            True            True               0     OK
     Consulta normal           amor tragedia vida           False           False               5     OK

Resultado global: 5/5 pruebas OK
Todas las pruebas de robustez pasaron correctamente.
